In [17]:
import requests


def get_wikipedia_page(title: str):
    """
    Retrieve the full text content of a Wikipedia page.

    :param title: str - Title of the Wikipedia page.
    :return: str - Full text content of the page as raw string.
    """
    # Wikipedia API endpoint
    URL = "https://ja.wikipedia.org/w/api.php"

    # Parameters for the API request
    params = {
        "action": "query",
        "format": "json",
        "titles": title,
        "prop": "extracts",
        "explaintext": True,
    }

    # Custom User-Agent header to comply with Wikipedia's best practices
    headers = {"User-Agent": "tutorial/0.0.1"}

    response = requests.get(URL, params=params, headers=headers)
    data = response.json()

    # Extracting page content
    page = next(iter(data["query"]["pages"].values()))
    return page["extract"] if "extract" in page else None

full_document = get_wikipedia_page("ネクセラファーマ")

#full_document


In [14]:
import os

def load_text_from_folder(folder_path: str) -> str:
    """
    指定フォルダ内の .txt ファイルをすべて読み込み、
    1つの文字列として結合して返す。

    :param folder_path: 読み込み対象フォルダのパス
    :return: 結合されたテキスト
    """
    texts = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".txt"):
            file_path = os.path.join(folder_path, filename)
            with open(file_path, "r", encoding="utf-8") as f:
                texts.append(f.read())

    return "\n".join(texts)


# 使い方（あなたの環境に合わせてパスを変更）
#folder_path = r"C:\Users\USER\Documents\RAG_system\semantic_chunk\txt"
#full_document = load_text_from_folder(folder_path)

#full_document


In [2]:
import torch
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

device = "cuda" if torch.cuda.is_available() else "cpu"

model_path = r"C:\Users\USER\Documents\RAG_system\semantic_chunk\models--BAAI--bge-m3"
model_kwargs = {"device": device}
encode_kwargs = {"normalize_embeddings": True}  # Cosine Similarity

embedding_model = HuggingFaceBgeEmbeddings(
    model_name=model_path,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)



C:\Users\USER\AppData\Local\Temp\ipykernel_21456\23068688.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceBgeEmbeddings
C:\Users\USER\AppData\Local\Temp\ipykernel_21456\23068688.py:10: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceBgeEmbeddings(
C:\Users\USER\miniforge3\envs\semantic_chunk\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See

In [67]:
import numpy as np
import re
from typing import List
from langchain_core.documents import Document


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


class HybridSemanticChunker:
    def __init__(self, embedding_model, breakpoint_ratio=0.15, merge_threshold=0.80):
        self.embedding_model = embedding_model
        self.breakpoint_ratio = breakpoint_ratio
        self.merge_threshold = merge_threshold

    # ① 段落分割（空行ベース）
    def _split_paragraphs(self, text: str) -> List[str]:
        paragraphs = re.split(r"\n\s*\n", text)
        return [p.strip() for p in paragraphs if p.strip()]

    # ② 文分割（句点 + 改行）
    def _split_sentences(self, paragraph: str) -> List[str]:
        sentences = re.split(r"(?<=[。．！？])\n+", paragraph)
        return [s.strip() for s in sentences if s.strip()]

    # ③ 埋め込み
    def _embed(self, items: List[str]) -> np.ndarray:
        return np.array(self.embedding_model.embed_documents(items))

    # ④ 類似度計算
    def _compute_similarities(self, embeddings: np.ndarray) -> List[float]:
        sims = [
            cosine_similarity(embeddings[i], embeddings[i - 1])
            for i in range(1, len(embeddings))
        ]
        return sims

    # ⑤ breakpoint 抽出（段落内は最大1つ）
    def _find_breakpoints(self, sims: List[float]) -> List[int]:
        if len(sims) == 0:
            return []
        threshold = np.quantile(sims, self.breakpoint_ratio)
        candidates = [i for i, s in enumerate(sims) if s < threshold]

        # 分割しすぎ防止：段落内は最大1つ
        return candidates[:1]

    # ⑥ ハイブリッド分割（分割しすぎ防止）
    def split_text(self, text: str) -> List[str]:
        paragraphs = self._split_paragraphs(text)
        final_chunks = []

        for para in paragraphs:
            sentences = self._split_sentences(para)

            if len(sentences) == 1:
                final_chunks.append(sentences[0])
                continue

            embeddings = self._embed(sentences)
            sims = self._compute_similarities(embeddings)
            breakpoints = self._find_breakpoints(sims)

            chunks = []
            current_chunk = sentences[0]

            for i in range(1, len(sentences)):
                sim = sims[i - 1]

                # 類似度が高い → 結合
                if sim > self.merge_threshold:
                    current_chunk += "\n" + sentences[i]
                    continue

                # 類似度が低い → breakpoint の場合だけ分割
                if (i - 1) in breakpoints:
                    chunks.append(current_chunk)
                    current_chunk = sentences[i]
                else:
                    current_chunk += "\n" + sentences[i]

            chunks.append(current_chunk)
            final_chunks.extend(chunks)

        return final_chunks

    # ⑦ Document 化
    def create_documents(self, texts: List[str]) -> List[Document]:
        docs = []
        for text in texts:
            for chunk in self.split_text(text):
                docs.append(Document(page_content=chunk))
        return docs


In [77]:
import numpy as np
import re
from typing import List, Dict
from langchain_core.documents import Document


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# ---------------------------------------------------------
# 0. 日本語見出し検出ロジック
# ---------------------------------------------------------
def detect_japanese_heading(line: str) -> bool:
    """
    日本語見出しの特徴:
    - 句読点がない
    - 40文字以内
    - 名詞句で終わる可能性が高い
    - 漢字・ひらがな・カタカナのみ
    """
    if len(line) == 0:
        return False

    # 句読点がある → 見出しではない
    if re.search(r"[。．、，！？]", line):
        return False

    # 長すぎる → 見出しではない
    if len(line) > 40:
        return False

    # 名詞で終わる可能性が高い語尾
    noun_suffixes = [
        "市場", "性質", "戦略", "概要", "説明", "動向", "展開",
        "紹介", "背景", "課題", "目的", "要点"
    ]
    if any(line.endswith(suffix) for suffix in noun_suffixes):
        return True

    # 漢字・ひらがな・カタカナのみ → 見出しの可能性が高い
    if re.match(r"^[一-龥ぁ-んァ-ンー]+$", line):
        return True

    return False


# ---------------------------------------------------------
# 1. 日本語プレーンテキストを「見出し単位」で分割
# ---------------------------------------------------------
def parse_japanese_sections(text: str) -> List[Dict]:
    lines = text.split("\n")

    blocks = []
    buffer = []
    current_section = "導入"

    def flush():
        nonlocal buffer, current_section
        if buffer:
            blocks.append({
                "section": current_section,
                "content": "\n".join(buffer).strip()
            })
            buffer = []

    for raw in lines:
        line = raw.strip()
        if not line:
            continue

        # 見出し判定
        if detect_japanese_heading(line):
            flush()
            current_section = line
            continue

        buffer.append(line)

    flush()
    return blocks


# ---------------------------------------------------------
# 2. HybridSemanticChunker（あなたの元コード）
# ---------------------------------------------------------
class HybridSemanticChunker:
    def __init__(self, embedding_model, breakpoint_ratio=0.15, merge_threshold=0.80):
        self.embedding_model = embedding_model
        self.breakpoint_ratio = breakpoint_ratio
        self.merge_threshold = merge_threshold

    def _split_paragraphs(self, text: str) -> List[str]:
        paragraphs = re.split(r"\n\s*\n", text)
        return [p.strip() for p in paragraphs if p.strip()]

    def _split_sentences(self, paragraph: str) -> List[str]:
        sentences = re.split(r"(?<=[。．！？])\s*", paragraph)
        return [s.strip() for s in sentences if s.strip()]

    def _embed(self, items: List[str]) -> np.ndarray:
        return np.array(self.embedding_model.embed_documents(items))

    def _compute_similarities(self, embeddings: np.ndarray) -> List[float]:
        return [
            cosine_similarity(embeddings[i], embeddings[i - 1])
            for i in range(1, len(embeddings))
        ]

    def _find_breakpoints(self, sims: List[float]) -> List[int]:
        if len(sims) == 0:
            return []
        threshold = np.quantile(sims, self.breakpoint_ratio)
        candidates = [i for i, s in enumerate(sims) if s < threshold]
        return candidates[:1]

    def split_text(self, text: str) -> List[str]:
        paragraphs = self._split_paragraphs(text)
        final_chunks = []

        for para in paragraphs:
            sentences = self._split_sentences(para)

            if len(sentences) == 1:
                final_chunks.append(sentences[0])
                continue

            embeddings = self._embed(sentences)
            sims = self._compute_similarities(embeddings)
            breakpoints = self._find_breakpoints(sims)

            chunks = []
            current_chunk = sentences[0]

            for i in range(1, len(sentences)):
                sim = sims[i - 1]

                if sim > self.merge_threshold:
                    current_chunk += "\n" + sentences[i]
                    continue

                if (i - 1) in breakpoints:
                    chunks.append(current_chunk)
                    current_chunk = sentences[i]
                else:
                    current_chunk += "\n" + sentences[i]

            chunks.append(current_chunk)
            final_chunks.extend(chunks)

        return final_chunks

    # ---------------------------------------------------------
    # 3. 日本語見出し単位でチャンク化
    # ---------------------------------------------------------
    def split_by_toc(self, text: str) -> List[Dict]:
        toc_blocks = parse_japanese_sections(text)
        results = []

        for block in toc_blocks:
            chunks = self.split_text(block["content"])
            for idx, chunk in enumerate(chunks):
                results.append({
                    "section": block["section"],
                    "chunk_index": idx,
                    "content": chunk
                })

        return results

    # ---------------------------------------------------------
    # 4. Document 化
    # ---------------------------------------------------------
    def create_documents(self, text) -> List[Document]:
        if isinstance(text, list):
            text = "\n".join(text)

        toc_chunks = self.split_by_toc(text)
        docs = []

        for c in toc_chunks:
            docs.append(
                Document(
                    page_content=c["content"],
                    metadata={
                        "section": c["section"],
                        "chunk_index": c["chunk_index"],
                    }
                )
            )

        return docs


In [79]:
from pprint import pprint

folder_path = r"C:\Users\USER\Documents\RAG_system\semantic_chunk\txt"
full_document = load_text_from_folder(folder_path)

chunker = HybridSemanticChunker(embedding_model, breakpoint_ratio=0.15, merge_threshold=0.80)

docs = chunker.create_documents([full_document])

for d in docs:
    print(d.page_content)
    print("+++" * 30)


FX
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
マガジンTOP
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
NISA・つみたてNISA
iDeCo（イデコ）
株 初心者入門
投資信託 初心者入門
FX 初心者入門
仮想通貨 初心者入門
不動産 初心者入門
先物取引 初心者入門
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
30日間無料で体験
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
学習/お得情報
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
PR
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
プロ厳選・推奨株
テンバガー★2026
今知りたい金・白金
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
株式/指数	 ニュース	 【QAあり】ネクセラファーマ、プラットフォーム型とパイプライン型の2事業を発展　高収益なバイオファーマを目指す
【QAあり】ネクセラファーマ、プラットフォーム型とパイプライン型の2事業を発展　高収益なバイオファーマを目指す
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
